**EXPLORATORY DATA ANALYSIS**

In [25]:
import pandas as pd 
import numpy as np

**PITSTOPS**

In [26]:
df_p=pd.read_parquet(r"C:\Users\Amitava\Downloads\Formula1\All_Data\Bronze_data\pit_stops\all_seasons_pit_stops.parquet")

In [27]:
df_p.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,stop_number,lap_number,local_time,duration,duration_ms,total_stops_race,avg_duration_ms_race
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,alonso,1,26,16:53:09,22.573,22573.0,1,22573.0
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,bottas,1,25,16:51:21,21.664,21664.0,1,21664.0
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,brendon_hartley,1,1,16:15:04,22.213,22213.0,2,22254.5
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,brendon_hartley,2,22,16:47:37,22.296,22296.0,2,22254.5
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,grosjean,1,24,16:49:31,23.054,23054.0,1,23054.0


In [28]:
df_p.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5941 entries, 0 to 5940
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   season                5941 non-null   int64  
 1   round_number          5941 non-null   int64  
 2   race_name             5941 non-null   object 
 3   circuit_ref           5941 non-null   object 
 4   race_date             5941 non-null   object 
 5   driver_ref            5941 non-null   object 
 6   stop_number           5941 non-null   Int16  
 7   lap_number            5941 non-null   Int16  
 8   local_time            5941 non-null   object 
 9   duration              5941 non-null   object 
 10  duration_ms           5934 non-null   float64
 11  total_stops_race      5941 non-null   Int16  
 12  avg_duration_ms_race  5940 non-null   float64
dtypes: Int16(3), float64(2), int64(2), object(6)
memory usage: 516.5+ KB


In [29]:
df_p['race_date']=pd.to_datetime(df_p['race_date'])

* Dropping **avg_duration_ms_race** because anomalies occur when during periods of driver crashes, heavy rain, faulty vehicle, etc when drivers need to wait at their pitstops, therefore their waiting time is recorded which actually is not required in prediction and also hampers the average time distribution
* Dropping **total_stops_race** is repetative column as **stop_number** already exists
* Dropping **local_time**, as local does not add any importance and lap timings have already been tracked

In [30]:
df_p.drop(columns={'avg_duration_ms_race','total_stops_race','local_time'},inplace=True)

As there is no **primary key** present therefore we create **candifate key p_ref** which will act as this table's **primary key** and checking whether combo of driver_ref, season, round_no, stop_number has any duplicates or not

In [31]:
pk_test = df_p.duplicated(
    subset=["driver_ref", "season", "round_number","stop_number"]
).sum()
print(f"Duplicates with driver_ref + season + round_number + stop_number: {pk_test}")

Duplicates with driver_ref + season + round_number + stop_number: 0


In [32]:
df_p["p_ref"]=df_p["driver_ref"]+"_"+df_p["season"].astype(str)+ "_" +df_p["round_number"].astype(str)

In [33]:
df_p[df_p['duration_ms'].isnull()]

,season,round_number,race_name,circuit_ref,race_date,driver_ref,stop_number,lap_number,duration,duration_ms,p_ref
241,2018,10,British Grand Prix,silverstone,2018-07-08,max_verstappen,2,33,,NaN,max_verstappen_2018_10
927,2019,13,Belgian Grand Prix,spa,2019-09-01,hulkenberg,2,31,,NaN,hulkenberg_2019_13
1028,2019,17,Japanese Grand Prix,suzuka,2019-10-13,bottas,1,17,,NaN,bottas_2019_17
1759,2020,17,Abu Dhabi Grand Prix,yas_marina,2020-12-13,gasly,1,10,,NaN,gasly_2020_17
2109,2021,9,Austrian Grand Prix,red_bull_ring,2021-07-04,mazepin,1,27,,NaN,mazepin_2021_9
4733,2024,12,British Grand Prix,silverstone,2024-07-07,tsunoda,1,27,,NaN,tsunoda_2024_12
5570,2025,11,Austrian Grand Prix,red_bull_ring,2025-06-29,tsunoda,2,30,,NaN,tsunoda_2025_11


Manually filling empty columns

In [34]:
import re
manual_fills = [
    {"driver_ref": "max_verstappen", "season": 2018, "round_number": 10, "stop_number": 2,  "duration": "28.000"},
    {"driver_ref": "hulkenberg",     "season": 2019, "round_number": 13, "stop_number": 2,  "duration": "23.000"},
    {"driver_ref": "bottas",         "season": 2019, "round_number": 17, "stop_number": 1,  "duration": "23.000"},
    {"driver_ref": "gasly",          "season": 2020, "round_number": 17, "stop_number": 1,  "duration": "22.000"},
    {"driver_ref": "mazepin",        "season": 2021, "round_number": 9,  "stop_number": 1,  "duration": "22.000"},
    {"driver_ref": "tsunoda",        "season": 2024, "round_number": 12, "stop_number": 1,  "duration": "30.000"},
    {"driver_ref": "tsunoda",        "season": 2025, "round_number": 11, "stop_number": 2,  "duration": "30.000"},
]

for fill in manual_fills:
    mask = (
        (df_p["driver_ref"]   == fill["driver_ref"]) &
        (df_p["season"]       == fill["season"])     &
        (df_p["round_number"] == fill["round_number"]) &
        (df_p["stop_number"]   == fill["stop_number"])
    )
    df_p.loc[mask, "duration"] = fill["duration"]

# conversion of duration to miliseconds 
def parse_pitstop_duration_to_ms(duration_str):
    if pd.isna(duration_str) or str(duration_str).strip() in ("", "None", "0.0", "0"):
        return None
    
    duration_str = str(duration_str).strip()
    
    # Format 1 — MM:SS.mmm
    m = re.fullmatch(r"(\d+):(\d{2})\.(\d+)", duration_str)
    if m:
        minutes = int(m.group(1))
        seconds = int(m.group(2))
        millis  = int(m.group(3).ljust(3, "0")[:3])
        return (minutes * 60 + seconds) * 1000 + millis

    # Format 2 — SS.mmm
    m = re.fullmatch(r"(\d+)\.(\d+)", duration_str)
    if m:
        seconds = int(m.group(1))
        millis  = int(m.group(2).ljust(3, "0")[:3])
        return seconds * 1000 + millis

    # Format 3 — no decimal
    m = re.fullmatch(r"(\d+)", duration_str)
    if m:
        return int(m.group(1)) * 1000
    return None
df_p["duration_ms"] = df_p["duration"].apply(parse_pitstop_duration_to_ms).astype("float64")

In [35]:
df_p.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5941 entries, 0 to 5940
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   season        5941 non-null   int64         
 1   round_number  5941 non-null   int64         
 2   race_name     5941 non-null   object        
 3   circuit_ref   5941 non-null   object        
 4   race_date     5941 non-null   datetime64[ns]
 5   driver_ref    5941 non-null   object        
 6   stop_number   5941 non-null   Int16         
 7   lap_number    5941 non-null   Int16         
 8   duration      5941 non-null   object        
 9   duration_ms   5941 non-null   float64       
 10  p_ref         5941 non-null   object        
dtypes: Int16(2), datetime64[ns](1), float64(1), int64(2), object(5)
memory usage: 452.7+ KB


In [36]:
df_p.to_parquet(r"C:\Users\Amitava\Downloads\Formula1\All_Data\Silver_data\cleaned_pitstop")